# DRC CoRL 2026 — Session 1: LIBERO (Part B)

Accelerator: **GPU T4 x2** (Settings → Accelerator). Trains Diffusion Policy + ACT on the four
LIBERO tasks, then computes the 8 offline metrics and 20 rollouts per checkpoint, pruning
checkpoints per task to stay under the 20 GB cap. ~2–8h depending on 1 vs 2 T4s and seed count.

In [ ]:
REPO_URL = "https://github.com/keshavkrishnan08/CORL.git"   # <-- set this
BRANCH   = "main"
%cd /kaggle/working
![ -d CORL ] && (cd CORL && git pull) || git clone -b $BRANCH $REPO_URL CORL
%cd /kaggle/working/CORL

In [ ]:
# confirm BOTH T4s are visible (a single process only uses cuda:0; the scripts launch one per GPU)
import torch
print('GPUs visible to torch:', torch.cuda.device_count(), '<-- expect 2 for T4 x2')
assert torch.cuda.is_available(), 'No GPU: set Accelerator to GPU T4 x2'

In [ ]:
# Install the LIBERO+Robomimic sim stack from source (NOT pip -r; see kaggle_setup.sh).
# TIP: run this cell ALONE first and confirm the import check passes before training.
!bash scripts/kaggle_setup.sh


### Validate the install (run this; **STOP and fix** if it fails)
Imports the sim stack and builds one real LIBERO env. Cheap insurance against a wasted training session.

In [ ]:
!MUJOCO_GL=egl python scripts/check_install.py < /dev/null


In [ ]:
# NOTE: data is downloaded PER TASK inside run_session (download -> train -> eval ->
# delete), so peak disk stays a few GB. Do NOT bulk-download the benchmark (~880GB).
# Record the pre-registration hash before training:
!python -c "from drc.utils import sha256_file; print('analysis.py', sha256_file('drc/analysis.py')); print('config.py', sha256_file('drc/config.py'))"


In [ ]:
# Session 1: train -> metrics -> rollouts -> prune, per task, across all visible GPUs.
# For a single T4, prepend SEEDS="0 1" to use 2 seeds.
!bash scripts/run_session1.sh
# (or:  !SEEDS="0 1" bash scripts/run_session1.sh)

In [ ]:
# checkpoint: LIBERO rows are now in results/metrics.csv + results/rollouts.csv
import pandas as pd
print(pd.read_csv('results/rollouts.csv').groupby('task')['success_rate'].mean())
# Save results/ as a Kaggle dataset output so Session 2 can pick it up (or push to GitHub).